In [2]:
import os
import pydicom
import numpy as np

# --- CONFIGURATION ---
# We focus on the first failing patient to diagnose the root cause
patient_path = '../Data/AMCGH/1042new'

print(f"--- DIAGNOSTIC REPORT: {patient_path} ---")

# 1. Find RTStruct and its Link
rt_struct_path = None
rt_referenced_uid = None

print("\n[1] SEARCHING FOR RTSTRUCT...")
for root, dirs, files in os.walk(patient_path):
    for f in files:
        path = os.path.join(root, f)
        try:
            dcm = pydicom.dcmread(path, stop_before_pixels=True, force=True)
            if dcm.get("Modality") == 'RTSTRUCT':
                rt_struct_path = path
                print(f"  Found RTStruct: {f}")
                
                # Try to find what it references
                try:
                    # Method A: Referenecd Frame of Reference
                    rfor = dcm.ReferencedFrameOfReferenceSequence[0]
                    study = rfor.RTReferencedStudySequence[0]
                    series = study.RTReferencedSeriesSequence[0]
                    rt_referenced_uid = series.SeriesInstanceUID
                    print(f"  -> Linked to CT Series UID: {rt_referenced_uid}")
                except:
                    print("  -> Could not extract Linked Series UID from header.")
                break
        except: continue
    if rt_struct_path: break

if not rt_struct_path:
    print("  [CRITICAL FAIL] No RTStruct found.")
    
# 2. Audit CT Files
print("\n[2] AUDITING CT IMAGES...")
ct_series = {} # {uid: {'count': 0, 'z_min': 999, 'z_max': -999}}

for root, dirs, files in os.walk(patient_path):
    for f in files:
        path = os.path.join(root, f)
        try:
            dcm = pydicom.dcmread(path, stop_before_pixels=True, force=True)
            if dcm.get("Modality") == 'CT':
                uid = dcm.SeriesInstanceUID
                z = float(dcm.ImagePositionPatient[2])
                
                if uid not in ct_series:
                    ct_series[uid] = {'count': 0, 'z_min': 999.0, 'z_max': -999.0}
                
                ct_series[uid]['count'] += 1
                if z < ct_series[uid]['z_min']: ct_series[uid]['z_min'] = z
                if z > ct_series[uid]['z_max']: ct_series[uid]['z_max'] = z
        except: continue

# 3. Report Findings
print("\n[3] CT SERIES FOUND IN FOLDER:")
match_found = False
for uid, info in ct_series.items():
    is_match = (uid == rt_referenced_uid)
    status = "MATCH! ✅" if is_match else "Mismatch ❌"
    if is_match: match_found = True
    
    print(f"  Series UID: {uid}")
    print(f"    - Files: {info['count']}")
    print(f"    - Z-Range: {info['z_min']:.1f} to {info['z_max']:.1f}")
    print(f"    - Status: {status}")

# 4. Check Contours (if struct exists)
if rt_struct_path:
    print("\n[4] CHECKING CONTOUR Z-POSITIONS...")
    try:
        from dicompylercore import dicomparser
        rtss = dicomparser.DicomParser(rt_struct_path)
        structures = rtss.GetStructures()
        
        # Find a tumor
        target_roi = None
        for k, v in structures.items():
            if 'gtv' in v['name'].lower():
                target_roi = k
                print(f"  Found ROI: {v['name']}")
                break
        
        if target_roi:
            contours = rtss.GetStructureCoordinates(target_roi)
            zs = sorted([float(z) for z in contours.keys()])
            if zs:
                print(f"  Contour Z-Range: {min(zs):.1f} to {max(zs):.1f}")
                
                # Check overlap with ALL series
                print("  Overlap Check:")
                for uid, info in ct_series.items():
                    # Check if contour is roughly inside image range
                    if min(zs) >= info['z_min'] - 5 and max(zs) <= info['z_max'] + 5:
                        print(f"    - Fits inside Series {uid[:10]}... ✅")
                    else:
                        print(f"    - Outside Series {uid[:10]}... ❌")
            else:
                print("  ROI has no coordinates.")
        else:
            print("  No GTV found to check.")
            
    except Exception as e:
        print(f"  Error parsing contours: {e}")

print("\n--- DIAGNOSIS COMPLETE ---")

--- DIAGNOSTIC REPORT: ../Data/AMCGH/1042new ---

[1] SEARCHING FOR RTSTRUCT...
  Found RTStruct: 20251042_StrctrSets.dcm
  -> Linked to CT Series UID: 1.2.392.200036.9116.2.6.1.37.2417525631.1750215275.404462

[2] AUDITING CT IMAGES...

[3] CT SERIES FOUND IN FOLDER:
  Series UID: 1.2.392.200036.9116.2.6.1.37.2417525631.1750215275.404462
    - Files: 133
    - Z-Range: -1138.5 to -478.5
    - Status: MATCH! ✅

[4] CHECKING CONTOUR Z-POSITIONS...
  Found ROI: GTV_3600/18
  Contour Z-Range: -883.5 to -793.5
  Overlap Check:
    - Fits inside Series 1.2.392.20... ✅

--- DIAGNOSIS COMPLETE ---
